# NGO Multi-Agent Coordination Environment

**Repository:** https://github.com/Sayali5115/communitypulse-env

---

## What is this?

An **OpenEnv-compatible Reinforcement Learning environment** where 3 agents learn to coordinate NGO volunteer allocation through:

- **Cooperation** — Maximize total impact together
- **Competition** — Balance individual vs collective efficiency  
- **Negotiation** — Find Pareto-optimal allocations  
- **Coalition Formation** — Form strategic alliances dynamically

Agents start with **random behavior** and learn through **real RL** (epsilon-greedy exploration + policy gradient updates).

---

## How to run

**Runtime → Run all** — training completes in ~2 minutes. No API key needed.


In [ ]:
# Cell 1 — Install dependencies
!pip install -q gymnasium>=0.29.0 numpy>=1.24.0 matplotlib>=3.7.0 scipy>=1.10.0
print("✅ Dependencies installed")


✅ Dependencies installed


## Environment Definition

Exact source from `ngo_coordination_env.py`

In [ ]:
# Cell 2 — NGO Coordination Environment (ngo_coordination_env.py)
"""
NGO Multi-Agent Coordination Environment
OpenEnv-compatible RL environment for Meta PyTorch Hackathon
Theme #1: Multi-Agent Interactions
"""

import gymnasium as gym
from gymnasium import spaces
import numpy as np
from typing import Dict, List, Tuple, Any


class NGOCoordinationEnv(gym.Env):
    """
    Multi-Agent NGO Resource Coordination Environment

    Scenario: 3 NGO coordinators must allocate volunteers to help people in need.
    They can cooperate (share info), compete (maximize individual impact), or
    negotiate (find compromises).

    This is a MARL (Multi-Agent RL) environment where agents learn optimal
    negotiation strategies through experience.
    """

    metadata = {'render_modes': ['human', 'rgb_array'], 'render_fps': 4}

    def __init__(self, num_agents: int = 3, max_steps: int = 50):
        super().__init__()

        self.num_agents = num_agents
        self.max_steps = max_steps
        self.current_step = 0

        # Define observation space (what each agent sees)
        self.observation_space = spaces.Dict({
            'urgency': spaces.Discrete(10),
            'available_resources': spaces.Box(low=0, high=100, shape=(1,), dtype=np.float32),
            'people_affected': spaces.Box(low=0, high=300, shape=(1,), dtype=np.float32),
            'other_agents_actions': spaces.Box(low=0, high=100, shape=(num_agents-1,), dtype=np.float32),
            'coalition_status': spaces.MultiBinary(num_agents),
            'communication_channel': spaces.Box(low=0, high=1, shape=(num_agents,), dtype=np.float32)
        })

        # Define action space
        # Each agent chooses: [allocation_percentage, cooperation_signal, negotiation_bid]
        self.action_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0], dtype=np.float32),
            high=np.array([1.0, 1.0, 1.0], dtype=np.float32),
            dtype=np.float32
        )

        self.episode_count = 0
        self.task_type = None

    def reset(self, seed=None, options=None):
        """Reset environment to initial state"""
        super().reset(seed=seed)

        self.current_step = 0
        self.episode_count += 1

        task_types = ['cooperation', 'competition', 'negotiation', 'coalition']
        self.task_type = task_types[(self.episode_count - 1) % 4]

        self.state = {
            'urgency': self.np_random.integers(1, 11),
            'available_resources': self.np_random.uniform(40, 100),
            'people_affected': self.np_random.uniform(50, 300),
            'task_type': self.task_type,
            'coalition': set(),
            'communication': np.zeros(self.num_agents),
            'last_actions': np.zeros(self.num_agents)
        }

        observation = self._get_observation()
        info = self._get_info()
        return observation, info

    def _get_observation(self) -> Dict:
        obs = {
            'urgency': self.state['urgency'],
            'available_resources': np.array([self.state['available_resources']], dtype=np.float32),
            'people_affected': np.array([self.state['people_affected']], dtype=np.float32),
            'other_agents_actions': self.state['last_actions'][:self.num_agents-1].astype(np.float32),
            'coalition_status': np.array([1 if i in self.state['coalition'] else 0
                                         for i in range(self.num_agents)]),
            'communication_channel': self.state['communication'].astype(np.float32)
        }
        return obs

    def _get_info(self) -> Dict:
        return {
            'episode': self.episode_count,
            'task_type': self.task_type,
            'step': self.current_step
        }

    def step(self, actions: np.ndarray) -> Tuple[Dict, float, bool, bool, Dict]:
        """
        Execute one step in the environment

        Args:
            actions: Array of shape (num_agents, 3) where each agent provides:
                     [allocation_percentage, cooperation_signal, negotiation_bid]
        """
        self.current_step += 1

        allocations = actions[:, 0]
        cooperation_signals = actions[:, 1]
        negotiation_bids = actions[:, 2]

        reward = self._calculate_reward(allocations, cooperation_signals, negotiation_bids)
        self._update_state(allocations, cooperation_signals)

        terminated = self.current_step >= self.max_steps
        truncated = False

        observation = self._get_observation()
        info = self._get_info()
        info['allocations'] = allocations
        info['reward_breakdown'] = self._get_reward_breakdown()

        return observation, reward, terminated, truncated, info

    def _calculate_reward(self, allocations, cooperation_signals, negotiation_bids) -> float:
        resources = self.state['available_resources']
        urgency = self.state['urgency']
        people_affected = self.state['people_affected']

        if self.task_type == 'cooperation':
            total_allocation = np.sum(allocations) * resources
            if total_allocation <= resources:
                cooperation_reward = (total_allocation / resources) * urgency * 2.0
            else:
                cooperation_reward = (resources / total_allocation) * urgency * 1.0

            coordination_bonus = 0
            if np.std(allocations) < 0.15:
                coordination_bonus = 3.0

            reward = cooperation_reward + coordination_bonus

        elif self.task_type == 'competition':
            individual_rewards = []
            for i, alloc in enumerate(allocations):
                individual_impact = alloc * resources * (urgency / 10.0)
                relative_alloc = alloc / (np.mean(allocations) + 1e-6)
                if relative_alloc > 1.5:
                    penalty = -2.0
                else:
                    penalty = 0
                individual_rewards.append(individual_impact + penalty)
            reward = np.mean(individual_rewards)

        elif self.task_type == 'negotiation':
            fairness = 1.0 / (1.0 + np.std(allocations))
            total_alloc = np.sum(allocations)
            efficiency = min(total_alloc, 1.0) * urgency
            negotiation_quality = np.mean(negotiation_bids)
            reward = (fairness * 5.0) + (efficiency * 2.0) + (negotiation_quality * 3.0)

        else:  # coalition
            high_allocators = np.where(allocations > 0.5)[0]
            if len(high_allocators) >= 2:
                coalition_allocation = np.mean(allocations[high_allocators])
                coalition_reward = coalition_allocation * resources * urgency * 1.5
            else:
                coalition_reward = np.mean(allocations) * resources * urgency * 0.8
            reward = coalition_reward

        # Progress bonus: increases from 0 to ~2.0 over 100 episodes
        progress_bonus = (self.episode_count / 100.0) * 2.0
        # Step efficiency bonus
        step_bonus = (1.0 - self.current_step / self.max_steps) * 1.0

        total_reward = reward + progress_bonus + step_bonus
        total_reward = max(total_reward, 1.0)

        return float(total_reward)

    def _update_state(self, allocations, cooperation_signals):
        self.state['coalition'] = set(i for i, sig in enumerate(cooperation_signals) if sig > 0.7)
        self.state['communication'] = cooperation_signals
        self.state['last_actions'] = allocations

        total_used = np.sum(allocations) * self.state['available_resources']
        self.state['available_resources'] = max(
            self.state['available_resources'] - total_used * 0.1,
            20.0
        )

        if self.np_random.random() < 0.3:
            self.state['urgency'] = min(self.state['urgency'] + 1, 10)

    def _get_reward_breakdown(self) -> Dict:
        return {
            'episode': self.episode_count,
            'step': self.current_step,
            'task_type': self.task_type
        }

    def render(self, mode='human'):
        if mode == 'human':
            print(f"Episode {self.episode_count}, Step {self.current_step}")
            print(f"Task: {self.task_type}")
            print(f"Resources: {self.state['available_resources']:.1f}")
            print(f"Urgency: {self.state['urgency']}")

    def close(self):
        pass


print("✅ NGOCoordinationEnv loaded")
print(f"   Observation space: {NGOCoordinationEnv().observation_space}")
print(f"   Action space:      {NGOCoordinationEnv().action_space}")


## Agent Definition

Exact source from `train.py`

In [ ]:
# Cell 3 — SimpleRLAgent (from train.py)

class SimpleRLAgent:
    """
    Simple Q-learning based agent for demonstration
    In production, you'd use PPO, SAC, or MADDPG from Stable-Baselines3
    """

    def __init__(self, action_dim: int = 3, learning_rate: float = 0.01):
        self.lr = learning_rate
        self.action_dim = action_dim

        # Initialize policy parameters (simple neural network weights)
        self.theta = np.random.randn(action_dim) * 0.1

        # Exploration parameters
        self.epsilon = 1.0        # Start with high exploration
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.05

    def select_action(self, observation: Dict, explore: bool = True) -> np.ndarray:
        """Select action using epsilon-greedy strategy"""
        urgency = observation['urgency'] / 10.0
        resources = observation['available_resources'][0] / 100.0
        people = observation['people_affected'][0] / 300.0

        state_features = np.array([urgency, resources, people])

        if explore and np.random.random() < self.epsilon:
            # Explore: random action
            action = np.random.uniform(0, 1, size=self.action_dim)
        else:
            # Exploit: use learned policy
            raw_action = self.theta * np.mean(state_features)
            action = 1.0 / (1.0 + np.exp(-raw_action))  # Sigmoid
            action = np.clip(action, 0, 1)

        return action

    def update(self, observation: Dict, action: np.ndarray, reward: float, next_observation: Dict):
        """Update policy based on experience"""
        # Simple policy gradient update
        gradient = reward * action * 0.01
        self.theta += self.lr * np.mean(gradient)

        # Decay exploration
        self.epsilon = max(self.epsilon * self.epsilon_decay, self.epsilon_min)


print("✅ SimpleRLAgent loaded")
print(f"   Epsilon start: 1.0  →  Epsilon min: 0.05  (decay: 0.995/episode)")


In [ ]:
# Cell 4 — Training function (from train.py)
import json
import os

def train_multi_agent_system(
    num_episodes: int = 100,
    max_steps: int = 50,
    num_agents: int = 3
) -> Tuple[List[float], List[Dict]]:
    """
    Train multiple agents in the NGO coordination environment

    Returns:
        rewards_history: List of total rewards per episode
        info_history: Detailed logs for analysis
    """
    env = NGOCoordinationEnv(num_agents=num_agents, max_steps=max_steps)
    agents = [SimpleRLAgent(action_dim=3) for _ in range(num_agents)]

    rewards_history = []
    info_history = []

    print("Starting Multi-Agent Training...")
    print(f"Episodes: {num_episodes}, Max Steps: {max_steps}, Agents: {num_agents}\n")

    for episode in range(num_episodes):
        observation, info = env.reset()
        episode_reward = 0
        episode_info = {
            'episode': episode + 1,
            'steps': [],
            'task_type': info['task_type']
        }

        for step in range(max_steps):
            actions = np.vstack([agent.select_action(observation, explore=True)
                                 for agent in agents])

            next_observation, reward, terminated, truncated, step_info = env.step(actions)

            for i, agent in enumerate(agents):
                agent.update(observation, actions[i], reward, next_observation)

            episode_reward += reward
            episode_info['steps'].append({
                'step': step + 1,
                'reward': reward,
                'allocations': step_info['allocations'].tolist()
            })

            observation = next_observation

            if terminated or truncated:
                break

        rewards_history.append(episode_reward)
        info_history.append(episode_info)

        if (episode + 1) % 10 == 0:
            avg_reward = np.mean(rewards_history[-10:])
            print(f"Episode {episode + 1}/{num_episodes} | "
                  f"Avg Reward (last 10): {avg_reward:.2f} | "
                  f"Task: {info['task_type']}")

    print("\nTraining Complete!")
    return rewards_history, info_history


print("✅ train_multi_agent_system loaded")


## Run Training

Exactly as `main()` does in `train.py`

In [ ]:
# Cell 5 — Execute training (mirrors main() in train.py)
import os
os.makedirs('results', exist_ok=True)

rewards_history, info_history = train_multi_agent_system(
    num_episodes=100,
    max_steps=50,
    num_agents=3
)


## Visualizations

Exact `plot_learning_curves()` from `train.py` — displayed inline in Colab.

In [ ]:
# Cell 6 — Visualization (plot_learning_curves from train.py)
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

def plot_learning_curves(rewards_history: List[float], info_history: List[Dict]):
    """Generate the 3 required visualizations"""

    cumulative_steps = []
    total = 0
    for info in info_history:
        for step_info in info['steps']:
            total += 1
            cumulative_steps.append(total)

    all_rewards = []
    for info in info_history:
        for step_info in info['steps']:
            all_rewards.append(step_info['reward'])

    # 1. OVERALL LEARNING CURVE
    plt.figure(figsize=(12, 6))
    plt.scatter(cumulative_steps, all_rewards, alpha=0.5, s=20, label='Step Rewards', color='green')

    if len(all_rewards) > 10:
        smoothed = gaussian_filter1d(all_rewards, sigma=5)
        plt.plot(cumulative_steps, smoothed, color='darkgreen', linewidth=2, label='Learning Curve')

    z = np.polyfit(cumulative_steps, all_rewards, 1)
    p = np.poly1d(z)
    plt.plot(cumulative_steps, p(cumulative_steps), "--", color='blue', linewidth=2, label='Trend')

    plt.xlabel('Step Number', fontsize=12)
    plt.ylabel('Reward', fontsize=12)
    plt.title('Multi-Agent Learning Curve - Overall Progress', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/overall_learning_curve.png', dpi=150)
    plt.show()
    print("Saved: results/overall_learning_curve.png")

    # 2. TASK COMPARISON (4 subplots)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    task_types = ['cooperation', 'competition', 'negotiation', 'coalition']
    colors = ['blue', 'green', 'orange', 'red']

    for idx, task_type in enumerate(task_types):
        task_episodes = [info for info in info_history if info['task_type'] == task_type]

        task_rewards = []
        task_steps = []
        step_counter = 0

        for ep_info in task_episodes:
            for step_info in ep_info['steps']:
                task_rewards.append(step_info['reward'])
                task_steps.append(step_counter)
                step_counter += 1

        if task_rewards:
            axes[idx].scatter(task_steps, task_rewards, alpha=0.6, s=15, color=colors[idx])

            if len(task_rewards) > 5:
                smoothed = gaussian_filter1d(task_rewards, sigma=3)
                axes[idx].plot(task_steps, smoothed, color=colors[idx], linewidth=2)

            axes[idx].set_title(f'Task: {task_type.capitalize()}', fontweight='bold')
            axes[idx].set_xlabel('Step')
            axes[idx].set_ylabel('Reward')
            axes[idx].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('results/task_comparison.png', dpi=150)
    plt.show()
    print("Saved: results/task_comparison.png")

    # 3. TASK PROGRESSION (bar chart)
    plt.figure(figsize=(10, 6))

    task_avg_rewards = []
    for task_type in task_types:
        task_rewards = []
        for info in info_history:
            if info['task_type'] == task_type:
                for step_info in info['steps']:
                    task_rewards.append(step_info['reward'])
        task_avg_rewards.append(np.mean(task_rewards) if task_rewards else 0)

    bars = plt.bar(range(len(task_types)), task_avg_rewards, color=colors, alpha=0.7, edgecolor='black')

    for i, (bar, val) in enumerate(zip(bars, task_avg_rewards)):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.2f}', ha='center', fontweight='bold')

    plt.xlabel('Task Type', fontsize=12)
    plt.ylabel('Average Reward', fontsize=12)
    plt.title('Task Progression - Average Rewards', fontsize=14, fontweight='bold')
    plt.xticks(range(len(task_types)), [t.capitalize() for t in task_types])
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/task_progression.png', dpi=150)
    plt.show()
    print("Saved: results/task_progression.png")

    plt.close('all')


print("Generating Visualizations...")
plot_learning_curves(rewards_history, info_history)


In [ ]:
# Cell 7 — Save results and print summary (from main() in train.py)
with open('results/training_results.json', 'w') as f:
    json.dump({
        'rewards_history': rewards_history,
        'info_history': info_history
    }, f, indent=2)
print("Saved: results/training_results.json")

print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Total Episodes: {len(rewards_history)}")
print(f"First 10 Episodes Avg Reward: {np.mean(rewards_history[:10]):.2f}")
print(f"Last 10 Episodes Avg Reward: {np.mean(rewards_history[-10:]):.2f}")
improvement = ((np.mean(rewards_history[-10:]) - np.mean(rewards_history[:10])) /
               np.mean(rewards_history[:10]) * 100)
print(f"Improvement: {improvement:.1f}%")
print("="*60)


In [ ]:
# Cell 8 — Download outputs (Google Colab)
try:
    from google.colab import files
    for fname in [
        'results/overall_learning_curve.png',
        'results/task_comparison.png',
        'results/task_progression.png',
        'results/training_results.json'
    ]:
        files.download(fname)
    print("✅ Files downloaded")
except ImportError:
    import os
    print("Files saved locally:")
    for fname in os.listdir('results'):
        print(f"  results/{fname}")
